# Quickstart: tempo of sung recordings

This notebook runs `vocal_tempo.py` on a folder of audio files.

1. Set `FOLDER` to your audio folder (it starts with the synthetic test files).
2. Choose a preset: `"generic"` or `"lullaby"`. Override single settings in `OVERRIDES`.
3. Run all cells. A CSV `tempo_summary_<preset>.csv` is written next to your audio.

Settings are explained in `docs/METHOD.md`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # the repository folder
import vocal_tempo as vt
print("vocal_tempo", vt.__version__)

In [ ]:
FOLDER = r"../examples/test_audio"   # e.g. r"C:\Users\name\recordings"
PRESET = "generic"                   # "generic" or "lullaby"
OVERRIDES = {}                       # e.g. {"mu": 110, "tmax": 200, "sigma": 25, "tactus": False}

vt.settings(PRESET, **OVERRIDES)

In [ ]:
# create the synthetic test files if the folder does not exist yet
if not os.path.isdir(FOLDER) and FOLDER.endswith("test_audio"):
    import subprocess; subprocess.run([sys.executable, "../examples/self_test.py"], check=False)

df = vt.analyze_folder(FOLDER, preset=PRESET, save_beats=False, **OVERRIDES)
df[["Filename", "Reported_BPM", "RBT_BPM_final", "ACF_BPM_final", "Num_Beats", "RBT_Halved", "Duration_s"]]

## Look at one file

Onset envelope with the tracked beats. Useful to see whether the beats sit on
the sung notes or on every second note (an octave error).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa

FILE = os.path.join(FOLDER, df["Filename"].iloc[0])
s = vt.settings(PRESET, **OVERRIDES)
y, sr = vt.load_preemphasize(FILE, sr=s["sr"], preemph=s["preemph"])
env = vt.onset_envelope(y, sr, hop=s["hop"], smooth_frames=s["smooth_frames"])
res = vt.analyze_file(FILE, preset=PRESET, **OVERRIDES)

t = librosa.frames_to_time(np.arange(len(env)), sr=sr, hop_length=s["hop"])
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, env, lw=0.8, color="#52514e")
for b in res["Beat_Times_sec"]:
    ax.axvline(b, color="#2a78d6", lw=0.8, alpha=0.7)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Onset strength")
ax.set_title(f"{os.path.basename(FILE)}: {res['Reported_BPM']} BPM", loc="left")
plt.show()